# Adversarial RF Robustness — Google Colab Runner

**Workflow:** **Python code** comes from **GitHub** (you edit locally in Cursor, push, then the setup cell runs `git pull`). **Large datasets** stay on **Google Drive** under `data/raw/` — set `DRIVE_PROJECT` in the setup cell to match your Drive path.

**Runtime > Change runtime type > T4 GPU (or better)**

### Setup reminder

- Run the **Setup** cell first: it mounts Drive, clones/pulls into `/content/ew_project`, and sets `DATA_2016` on Drive. **Private repo:** in Colab use **Secrets** (key `GITHUB_TOKEN`, value = a [fine-grained PAT](https://github.com/settings/personal-access-tokens) with **Contents: Read** on `ew_project`), and enable **Notebook access** for that secret.
- Edit **`DRIVE_PROJECT`** in that cell so it matches the folder on Drive where `data/raw/` lives.
- The staged tuning workflow needs **both** `DATA_2016` and `DATA_2018`. It writes outputs under `experiments/tuning_workflow/` by default.

### Monitoring long runs

- **tqdm**: Evaluation scripts show a batch progress bar (phase, SNR, trial). Disable with `--no-progress` if logs get too noisy.
- **JSON status files** (updated as each sub-step finishes):
  - `evaluate.py` → `<output_dir>/eval_progress.json`
  - `eval_per_class.py` → `<output_dir>/eval_progress.json`
  - `eval_transferability.py` → `<output_dir>/transferability_progress.json`
  - `train_defenses.py` (defense comparison phase) → `<output_dir>/defense_eval_progress.json`
- **Workflow artifacts**:
  - `run_tuning_workflow.py` → `experiments/tuning_workflow/workflow_summary.csv`
  - `tune_defenses.py` → `experiments/tuning_workflow/defenses/defense_tuning/best_by_defense_defense_tuning.json`
- **Peek from another Colab cell** (while a job runs): `!cat experiments/results/baseline_cnn/eval_progress.json` or `!cat experiments/tuning_workflow/workflow_summary.csv` after tuning finishes.


In [7]:
# --- Setup: code from GitHub (/content), data files on Google Drive ---
# Run this cell first. Edit DRIVE_PROJECT to where your data/ folder lives on Drive.

from google.colab import drive

drive.mount("/content/drive")

import os
import shutil
import subprocess

# ---- 1) Clone or update repo (Python project lives in Colab VM, synced from Git) ----
GITHUB_USER = "Elijah3756"
REPO_NAME = "ew_project"
CODE_DIR = f"/content/{REPO_NAME}/adversarial-rf-robustness"

# Private repo: add GITHUB_TOKEN in Colab (Secrets, notebook access ON).
# Fine-grained PAT: Repository access + Contents: Read (or classic PAT with repo scope).
token = os.environ.get("GITHUB_TOKEN", "").strip()
if token:
    clone_url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
else:
    clone_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

repo_root = f"/content/{REPO_NAME}"
git_dir = os.path.join(repo_root, ".git")

if os.path.isdir(git_dir):
    r = subprocess.run(
        ["git", "-C", repo_root, "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    print((r.stdout or "").strip() or "git pull OK")
    if r.returncode != 0:
        print("git pull stderr:", r.stderr)
        raise RuntimeError("git pull failed; fix auth/network and re-run.")
elif os.path.exists(repo_root):
    print("Removing incomplete folder (no .git), then cloning fresh...")
    shutil.rmtree(repo_root)

if not os.path.isdir(git_dir):
    r = subprocess.run(
        ["git", "clone", "--depth", "1", clone_url, repo_root],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print("--- git stderr ---\n", r.stderr)
        print("--- git stdout ---\n", r.stdout)
        print(
            "\nClone failed (exit 128 usually = auth). If ew_project is PRIVATE, add "
            "GITHUB_TOKEN to Colab Secrets and re-run. Public repos clone without a token."
        )
        raise RuntimeError("git clone failed; see messages above.")

os.chdir(CODE_DIR)
print("Code directory:", os.getcwd())

# ---- 2) Drive path that contains data/raw/ (upload large files here only) ----
DRIVE_PROJECT = "/content/drive/MyDrive/ew_project-old/adversarial-rf-robustness"

DATA_2016 = os.path.join(DRIVE_PROJECT, "data/raw/RML2016.10a_dict.pkl")
_c2018 = [
    os.path.join(DRIVE_PROJECT, "data/raw/GOLD_XYZ_OSC.0001_1024.hdf5"),
    os.path.join(DRIVE_PROJECT, "data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5"),
]
DATA_2018 = next((p for p in _c2018 if os.path.isfile(p)), None)

if not os.path.isfile(DATA_2016):
    print("WARNING: 2016 dataset missing:", DATA_2016)
else:
    print("2016 data OK:", DATA_2016)
print("2018 data:", DATA_2018 or "(optional; upload HDF5 to data/raw/ on Drive)")

os.environ["DATA_2016"] = DATA_2016
if DATA_2018:
    os.environ["DATA_2018"] = DATA_2018

print("Setup done. Next: pip install, then GPU check.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
Code directory: /content/ew_project/adversarial-rf-robustness
2016 data OK: /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/RML2016.10a_dict.pkl
2018 data: /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5
Setup done. Next: pip install, then GPU check.


In [8]:
# Install Python deps (run the Setup cell first so `cwd` is the repo)
!pip install -q scipy pandas matplotlib h5py pyyaml tqdm scikit-learn


In [9]:
# 3. Verify GPU (PyTorch is preinstalled on Colab with CUDA)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


### Hyperparameter Tuning Workflow

Run the staged coarse-to-fine workflow before long experiment sweeps when you want data-driven hyperparameters instead of the notebook defaults.

- `run_tuning_workflow.py` requires both `DATA_2016` and `DATA_2018`
- Full outputs land under `experiments/tuning_workflow/`
- The reduced-grid smoke run below is only for fast validation; the full workflow is the authoritative source for tuned settings
- The later baseline/defense training cells now load tuned values from these workflow artifacts when they exist

In [10]:
# 4a. Tuning output locations
TUNING_ROOT = 'experiments/tuning_workflow'
TUNING_SMOKE_ROOT = 'experiments/tuning_workflow_smoke'

print('Tuning root:', TUNING_ROOT)
print('Smoke-test root:', TUNING_SMOKE_ROOT)
print('2016 data:', DATA_2016)
print('2018 data:', DATA_2018)
if DATA_2018 is None:
    print('Upload the 2018 HDF5 first before running the tuning workflow.')

Tuning root: experiments/tuning_workflow
Smoke-test root: experiments/tuning_workflow_smoke
2016 data: /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/RML2016.10a_dict.pkl
2018 data: /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5


In [11]:
# 4b. Preflight check for tuning workflow files
import json
import os
import subprocess
import time

#region agent log
DEBUG_LOG_PATH = '/Users/elijahbellamy/Desktop/ew_project/.cursor/debug-e85316.log'
DEBUG_SESSION_ID = 'e85316'

def _agent_log(message, data, hypothesis_id, run_id='colab-preflight'):
    payload = {
        'sessionId': DEBUG_SESSION_ID,
        'runId': run_id,
        'hypothesisId': hypothesis_id,
        'location': 'run_colab.ipynb:preflight',
        'message': message,
        'data': data,
        'timestamp': int(time.time() * 1000),
    }
    try:
        with open(DEBUG_LOG_PATH, 'a') as f:
            f.write(json.dumps(payload) + '\n')
    except Exception:
        pass
#endregion


def tuning_preflight(run_id='colab-preflight'):
    required_files = [
        'run_tuning_workflow.py',
        'tune_train.py',
        'tune_defenses.py',
    ]
    file_state = {name: os.path.isfile(name) for name in required_files}
    git_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True)
    repo_root = git_top.stdout.strip() if git_top.returncode == 0 else None
    repo_state = {
        'cwd': os.getcwd(),
        'repo_root': repo_root,
        'git_repo_present': git_top.returncode == 0,
        'data_2016_exists': os.path.isfile(DATA_2016),
        'data_2018_exists': DATA_2018 is not None and os.path.isfile(DATA_2018),
    }
    _agent_log('tuning_preflight_repo_state', repo_state, 'H2', run_id)
    _agent_log('tuning_preflight_required_files', file_state, 'H1', run_id)

    print('cwd:', repo_state['cwd'])
    print('repo root:', repo_state['repo_root'])
    print('git repo present:', repo_state['git_repo_present'])
    for name, exists in file_state.items():
        print(f'{name}:', exists)

    if not repo_state['git_repo_present']:
        raise RuntimeError('This notebook is not running from a git checkout. Re-run the Setup cell first.')

    if not repo_state['data_2016_exists'] or not repo_state['data_2018_exists']:
        raise RuntimeError('Dataset paths are not ready. Re-run the setup cell and verify DATA_2016 / DATA_2018 first.')

    missing = [name for name, exists in file_state.items() if not exists]
    if missing:
        git_status = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
        _agent_log('tuning_preflight_git_status', {'stdout': git_status.stdout, 'returncode': git_status.returncode}, 'H3', run_id)
        print('\nMissing tuning files in the Colab checkout:', ', '.join(missing))
        print('This usually means the latest tuning files are not in the repo clone yet.')
        print('Push the local tuning changes to GitHub, then re-run the Setup cell so Colab pulls the latest repo.')
        raise RuntimeError('Colab checkout is missing the tuning workflow files.')

    print('Tuning workflow preflight passed.')

In [ ]:
import subprocess
import threading
import queue
import time
from tqdm.auto import tqdm

tuning_preflight(run_id='smoke-preflight')

smoke_cmd = [
    "python", "run_tuning_workflow.py",
    "--data_path_2016", DATA_2016,
    "--data_path_2018", DATA_2018,
    "--output_root", TUNING_SMOKE_ROOT,
    "--coarse_lr", "0.001",
    "--coarse_batch_size", "64",
    "--coarse_epochs_2016", "2",
    "--coarse_epochs_2018", "2",
    "--coarse_weight_decay", "0.0001",
    "--weight_decay_followup", "0.0001",
    "--defense_rho", "0.01",
    "--defense_pgd_steps", "3",
    "--defense_sigma", "0.01",
    "--refine_top_k", "1",
    "--top_k_summary", "1",
    "--defense_shared_top_k", "1",
    "--refine_lr_factors", "1.0",
    "--refine_batch_multipliers", "1.0",
    "--refine_epoch_offsets", "0",
    "--refine_weight_decay_factors", "1.0",
    "--device", "auto",
]

print(" ".join(smoke_cmd))
process = subprocess.Popen(
    smoke_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

line_queue = queue.Queue()


def _enqueue_output(stream, q):
    for line in iter(stream.readline, ""):
        q.put(line)
    stream.close()


reader = threading.Thread(target=_enqueue_output, args=(process.stdout, line_queue), daemon=True)
reader.start()

last_stage = "starting"
with tqdm(desc="Smoke tuning workflow", unit="s") as pbar:
    while process.poll() is None or not line_queue.empty():
        drained = False
        while not line_queue.empty():
            drained = True
            line = line_queue.get()
            print(line, end="")
            if line.startswith("== Stage:"):
                last_stage = line.strip().replace("== ", "").replace(" ==", "")
                pbar.set_postfix_str(last_stage)
        if process.poll() is None:
            time.sleep(1)
            pbar.update(1)
        elif not drained:
            break

returncode = process.wait()
print("returncode:", returncode)

if returncode != 0:
    raise RuntimeError("Smoke run failed; see streamed output above.")

cwd: /content/ew_project/adversarial-rf-robustness
repo root: /content/ew_project
git repo present: True
run_tuning_workflow.py: True
tune_train.py: True
tune_defenses.py: True
Tuning workflow preflight passed.
python run_tuning_workflow.py --data_path_2016 /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/RML2016.10a_dict.pkl --data_path_2018 /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5 --output_root experiments/tuning_workflow_smoke --coarse_lr 0.001 --coarse_batch_size 64 --coarse_epochs_2016 2 --coarse_epochs_2018 2 --coarse_weight_decay 0.0001 --weight_decay_followup 0.0001 --defense_rho 0.01 --defense_pgd_steps 3 --defense_sigma 0.01 --refine_top_k 1 --top_k_summary 1 --defense_shared_top_k 1 --refine_lr_factors 1.0 --refine_batch_multipliers 1.0 --refine_epoch_offsets 0 --refine_weight_decay_factors 1.0 --device auto


Smoke tuning workflow: 0s [00:00, ?s/s]

Using device: cuda
Dataset: 2016.10a, SNR range: (-10, 18)
Train: 115,499 | Val: 24,750 | Test: 24,751
Model parameters: 126,475

Training for 2 epochs...
----------------------------------------------------------------------
Epoch   1/2 | Train Loss: 0.9362 Acc: 0.6046 | Val Loss: 0.7431 Acc: 0.6564 | LR: 0.000500 | Best: 0.6564 | Time: 19s
----------------------------------------------------------------------
Training complete in 38.3s (0.01 GPU hours)
Best validation accuracy: 0.7000

Skipping clean SNR sweep (--skip_snr_sweep).

Results saved to: experiments/tuning_workflow_smoke/baseline/baseline2016_coarse/lr0p001_bs64_ep2_wd0p0001/
=== baseline2016_coarse (1 runs) ===
[1/1] lr0p001_bs64_ep2_wd0p0001
Refine candidates: experiments/tuning_workflow_smoke/baseline/refine_candidates_baseline2016_coarse.json

--- Top baseline configurations ---
   lr  batch_size  epochs  weight_decay                 trial_tag                                                                             

In [13]:
import os
print("2016 exists:", os.path.isfile(DATA_2016), DATA_2016)
print("2018 exists:", os.path.isfile(DATA_2018), DATA_2018)

2016 exists: True /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/RML2016.10a_dict.pkl
2018 exists: True /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5


### Fast Hyperparameter Search (10% Stratified Subset)

Use `--subset_fraction 0.10` to train on only 10% of the data (stratified by class) during HP search.
This cuts each trial's training time ~10x while preserving the relative ranking between configurations.
Val/test splits remain full-sized so metric comparisons are reliable.

**After finding the best HP config, run the final training (Cells 14/27) on the full dataset with those values.**

In [ ]:
# 4f. Fast HP search using 10% stratified subset
# ~10x faster per trial; finds optimal lr, batch_size, epochs, weight_decay
import subprocess

SUBSET_FRACTION = 0.10  # Change to 0.15 or 0.30 if you want more data per trial
HP_SEARCH_ROOT = 'experiments/hp_search_fast'

fast_cmd = [
    'python', 'run_tuning_workflow.py',
    '--data_path_2016', DATA_2016,
    '--data_path_2018', DATA_2018,
    '--output_root', HP_SEARCH_ROOT,
    '--subset_fraction', str(SUBSET_FRACTION),
    '--device', 'cuda',
    '--coarse_lr', '0.003', '0.001', '0.0005', '0.0001',
    '--coarse_batch_size', '64', '128', '256',
    '--coarse_epochs_2016', '30',
    '--coarse_epochs_2018', '30',
    '--coarse_weight_decay', '0.0001',
    '--weight_decay_followup', '0.00001', '0.0001', '0.001',
    '--refine_top_k', '3',
    '--top_k_summary', '5',
    '--refine_lr_factors', '0.5', '1.0', '2.0',
    '--refine_batch_multipliers', '1.0',
    '--refine_epoch_offsets', '0', '10',
    '--refine_weight_decay_factors', '0.1', '1.0', '10.0',
    '--defense_shared_top_k', '1',
    '--defense_rho', '0.01',
    '--defense_pgd_steps', '3', '5',
    '--defense_sigma', '0.01', '0.02',
]

print(f'HP search with {SUBSET_FRACTION:.0%} of training data')
print(' '.join(fast_cmd))
subprocess.run(fast_cmd, check=True)

# Show results
import pandas as pd, os
summary = os.path.join(HP_SEARCH_ROOT, 'workflow_summary.csv')
if os.path.isfile(summary):
    print('\n=== Fast HP Search Results ===')
    display(pd.read_csv(summary))
else:
    print('No summary yet — check output above for errors.')

In [ ]:
# 4c. Full staged tuning workflow
import subprocess
import threading
import queue
import time
from tqdm.auto import tqdm

tuning_preflight(run_id='full-preflight')

full_cmd = [
    'python', 'run_tuning_workflow.py',
    '--data_path_2016', DATA_2016,
    '--data_path_2018', DATA_2018,
    '--output_root', TUNING_ROOT,
    '--device', 'cuda',
    '--coarse_lr', '0.001', '0.0003',
    '--coarse_batch_size', '128', '256',
    '--coarse_epochs_2016', '40', '80',
    '--coarse_epochs_2018', '60', '120',
    '--coarse_weight_decay', '0.0001',
    '--refine_top_k', '2',
    '--top_k_summary', '3',
    '--refine_lr_factors', '0.5', '1.0',
    '--refine_batch_multipliers', '1.0',
    '--refine_epoch_offsets', '0', '20',
    '--refine_weight_decay_factors', '1.0', '5.0',
    '--weight_decay_followup', '0.0001', '0.0005',
    '--defense_shared_top_k', '1',
    '--defense_rho', '0.01',
    '--defense_pgd_steps', '3', '5',
    '--defense_sigma', '0.01', '0.02',
]

print(" ".join(full_cmd))
process = subprocess.Popen(
    full_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

line_queue = queue.Queue()

def _enqueue_output(stream, q):
    for line in iter(stream.readline, ""):
        q.put(line)
    stream.close()

reader = threading.Thread(
    target=_enqueue_output,
    args=(process.stdout, line_queue),
    daemon=True,
)
reader.start()

last_stage = "starting"
with tqdm(desc="Full tuning workflow", unit="s") as pbar:
    while process.poll() is None or not line_queue.empty():
        drained = False
        while not line_queue.empty():
            drained = True
            line = line_queue.get()
            print(line, end="")
            if line.startswith("== Stage:"):
                last_stage = line.strip().replace("== ", "").replace(" ==", "")
                pbar.set_postfix_str(last_stage)
        if process.poll() is None:
            time.sleep(1)
            pbar.update(1)
        elif not drained:
            break

returncode = process.wait()
print("returncode:", returncode)

if returncode != 0:
    raise RuntimeError("Full run failed; see streamed output above.")

cwd: /content/ew_project/adversarial-rf-robustness
repo root: /content/ew_project
git repo present: True
run_tuning_workflow.py: True
tune_train.py: True
tune_defenses.py: True
Tuning workflow preflight passed.
python run_tuning_workflow.py --data_path_2016 /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/RML2016.10a_dict.pkl --data_path_2018 /content/drive/MyDrive/ew_project-old/adversarial-rf-robustness/data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5 --output_root experiments/tuning_workflow --device cuda --coarse_lr 0.001 0.0003 --coarse_batch_size 128 256 --coarse_epochs_2016 40 80 --coarse_epochs_2018 60 120 --coarse_weight_decay 0.0001 --refine_top_k 2 --top_k_summary 3 --refine_lr_factors 0.5 1.0 --refine_batch_multipliers 1.0 --refine_epoch_offsets 0 20 --refine_weight_decay_factors 1.0 5.0 --weight_decay_followup 0.0001 0.0005 --defense_shared_top_k 1 --defense_rho 0.01 --defense_pgd_steps 3 5 --defense_sigma 0.01 0.02


Full tuning workflow: 0s [00:00, ?s/s]

Using device: cuda
Dataset: 2016.10a, SNR range: (-10, 18)
Train: 115,499 | Val: 24,750 | Test: 24,751
Model parameters: 126,475

Training for 40 epochs...
----------------------------------------------------------------------
Epoch   1/40 | Train Loss: 0.9575 Acc: 0.6034 | Val Loss: 0.7691 Acc: 0.6480 | LR: 0.000998 | Best: 0.6480 | Time: 16s
Epoch   5/40 | Train Loss: 0.6689 Acc: 0.7118 | Val Loss: 0.6308 Acc: 0.7191 | LR: 0.000962 | Best: 0.7191 | Time: 79s
Epoch  10/40 | Train Loss: 0.6165 Acc: 0.7285 | Val Loss: 0.6159 Acc: 0.7191 | LR: 0.000854 | Best: 0.7272 | Time: 157s
Epoch  15/40 | Train Loss: 0.5780 Acc: 0.7418 | Val Loss: 0.6333 Acc: 0.7252 | LR: 0.000691 | Best: 0.7272 | Time: 235s
Epoch  20/40 | Train Loss: 0.5209 Acc: 0.7629 | Val Loss: 0.6339 Acc: 0.7286 | LR: 0.000500 | Best: 0.7286 | Time: 314s
Epoch  25/40 | Train Loss: 0.4488 Acc: 0.7873 | Val Loss: 0.7053 Acc: 0.7242 | LR: 0.000309 | Best: 0.7286 | Time: 392s
Epoch  30/40 | Train Loss: 0.3835 Acc: 0.8121 | Val Los

In [ ]:
# 4d. Inspect tuning results
import json
import os
import pandas as pd

summary_path = os.path.join(TUNING_ROOT, 'workflow_summary.csv')
defense_path = os.path.join(TUNING_ROOT, 'defenses', 'defense_tuning', 'best_by_defense_defense_tuning.json')

if os.path.isfile(summary_path):
    print('=== Workflow Summary ===')
    display(pd.read_csv(summary_path))
else:
    print('workflow_summary.csv not found yet. Run the full workflow cell first.')

if os.path.isfile(defense_path):
    print('\n=== Best Defense Configs ===')
    with open(defense_path, 'r') as f:
        display(pd.DataFrame.from_dict(json.load(f), orient='index'))
else:
    print('best_by_defense_defense_tuning.json not found yet.')

In [ ]:
# 4e. Load tuned configs for the later training cells
import json
import os
import subprocess

DEFAULT_BASELINE_2016_CFG = {
    'lr': 0.001,
    'batch_size': 256,
    'epochs': 60,
    'weight_decay': 1e-4,
}
DEFAULT_BASELINE_2018_CFG = {
    'lr': 0.001,
    'batch_size': 256,
    'epochs': 60,
    'weight_decay': 1e-4,
}
DEFAULT_DEFENSE_CFGS = {
    'channel_aug': {'lr': 0.001, 'batch_size': 256, 'epochs': 60, 'weight_decay': 1e-4},
    'adv_train': {'lr': 0.001, 'batch_size': 256, 'epochs': 60, 'weight_decay': 1e-4, 'rho': 0.01, 'pgd_steps': 5},
    'adv_train_channel': {'lr': 0.001, 'batch_size': 256, 'epochs': 60, 'weight_decay': 1e-4, 'rho': 0.01, 'pgd_steps': 5},
    'noise_inject': {'lr': 0.001, 'batch_size': 256, 'epochs': 60, 'weight_decay': 1e-4, 'sigma': 0.01},
    'noise_inject_channel': {'lr': 0.001, 'batch_size': 256, 'epochs': 60, 'weight_decay': 1e-4, 'sigma': 0.01},
}
DEFENSE_ORDER = [
    'channel_aug',
    'adv_train',
    'adv_train_channel',
    'noise_inject',
    'noise_inject_channel',
]


def _read_json(path):
    if not os.path.isfile(path):
        return None
    with open(path, 'r') as f:
        return json.load(f)


def _merge_cfg(default_cfg, tuned_cfg):
    cfg = default_cfg.copy()
    if tuned_cfg:
        for key, value in tuned_cfg.items():
            if value is not None:
                cfg[key] = value
    return cfg


def _load_top_row(search_name):
    path = os.path.join(TUNING_ROOT, 'baseline', f'top_{search_name}.json')
    rows = _read_json(path)
    if not rows:
        return None
    return rows[0]


def _can_use_shared_defense_command(defense_cfgs):
    shared_keys = ('lr', 'batch_size', 'epochs', 'weight_decay')
    shared_values = {
        tuple(cfg[key] for key in shared_keys)
        for cfg in defense_cfgs.values()
    }
    adv_values = {
        (defense_cfgs[name].get('rho'), defense_cfgs[name].get('pgd_steps'))
        for name in ('adv_train', 'adv_train_channel')
    }
    noise_values = {
        defense_cfgs[name].get('sigma')
        for name in ('noise_inject', 'noise_inject_channel')
    }
    return len(shared_values) == 1 and len(adv_values) == 1 and len(noise_values) == 1


def run_tuned_defense_training(*, data_path, dataset_version, num_classes, input_length, output_dir):
    if _can_use_shared_defense_command(DEFENSE_CFGS):
        shared_cfg = DEFENSE_CFGS['channel_aug']
        adv_cfg = DEFENSE_CFGS['adv_train']
        noise_cfg = DEFENSE_CFGS['noise_inject']
        cmd = [
            'python', 'train_defenses.py',
            '--data_path', data_path,
            '--dataset_version', dataset_version,
            '--num_classes', str(num_classes),
            '--input_length', str(input_length),
            '--epochs', str(shared_cfg['epochs']),
            '--batch_size', str(shared_cfg['batch_size']),
            '--lr', str(shared_cfg['lr']),
            '--weight_decay', str(shared_cfg['weight_decay']),
            '--seed', '42',
            '--rho', str(adv_cfg['rho']),
            '--pgd_steps', str(adv_cfg['pgd_steps']),
            '--sigma', str(noise_cfg['sigma']),
            '--output_dir', output_dir,
            '--eval_snr', '0', '10',
            '--device', 'auto',
        ]
        print('Shared tuned defense config detected; running one joint command:')
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)
        return

    print('Per-defense tuned settings differ; running defense training sequentially.')
    for i, defense_name in enumerate(DEFENSE_ORDER):
        cfg = DEFENSE_CFGS[defense_name]
        cmd = [
            'python', 'train_defenses.py',
            '--data_path', data_path,
            '--dataset_version', dataset_version,
            '--num_classes', str(num_classes),
            '--input_length', str(input_length),
            '--epochs', str(cfg['epochs']),
            '--batch_size', str(cfg['batch_size']),
            '--lr', str(cfg['lr']),
            '--weight_decay', str(cfg['weight_decay']),
            '--seed', '42',
            '--output_dir', output_dir,
            '--eval_snr', '0', '10',
            '--device', 'auto',
            '--defenses', defense_name,
        ]
        if cfg.get('rho') is not None:
            cmd.extend(['--rho', str(cfg['rho'])])
        if cfg.get('pgd_steps') is not None:
            cmd.extend(['--pgd_steps', str(cfg['pgd_steps'])])
        if cfg.get('sigma') is not None:
            cmd.extend(['--sigma', str(cfg['sigma'])])
        if i < len(DEFENSE_ORDER) - 1:
            cmd.append('--skip_defense_eval')

        print(' '.join(cmd))
        subprocess.run(cmd, check=True)


BASELINE_2016_CFG = _merge_cfg(
    DEFAULT_BASELINE_2016_CFG,
    _load_top_row('baseline2016_weight_decay')
    or _load_top_row('baseline2016_refine')
    or _load_top_row('baseline2016_coarse'),
)
BASELINE_2018_CFG = _merge_cfg(
    DEFAULT_BASELINE_2018_CFG,
    _load_top_row('baseline2018_refine')
    or _load_top_row('baseline2018_coarse'),
)

best_by_defense = _read_json(
    os.path.join(TUNING_ROOT, 'defenses', 'defense_tuning', 'best_by_defense_defense_tuning.json')
) or {}
DEFENSE_CFGS = {
    name: _merge_cfg(default_cfg, best_by_defense.get(name))
    for name, default_cfg in DEFAULT_DEFENSE_CFGS.items()
}

print('2016 baseline config:', BASELINE_2016_CFG)
print('2018 baseline config:', BASELINE_2018_CFG)
print('Defense configs loaded from tuning outputs:', sorted(best_by_defense.keys()))
print('Run this cell before the later baseline/defense training cells.')

In [ ]:
# 4. Run smoke test
!python smoke_test.py --data_path {DATA_2016}

In [ ]:
# 5. Phase 0: Baseline CNN Training (uses tuned params when available)
!python train.py \
  --data_path {DATA_2016} \
  --dataset_version 2016.10a \
  --epochs {BASELINE_2016_CFG['epochs']} \
  --batch_size {BASELINE_2016_CFG['batch_size']} \
  --lr {BASELINE_2016_CFG['lr']} \
  --weight_decay {BASELINE_2016_CFG['weight_decay']} \
  --seed 42 \
  --save_dir experiments/results/baseline_cnn \
  --device auto

In [ ]:
# 6. Phase 1: Clean Accuracy vs SNR
!python evaluate.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path {DATA_2016} \
  --eval_mode clean_snr \
  --num_trials 10 \
  --output_dir experiments/results/baseline_cnn \
  --device auto \
  --seed 42

In [ ]:
# 7. Phase 2: Attack evaluations (FGSM + PGD across channels)
for attack in ['fgsm', 'pgd']:
    for channel in ['awgn', 'rayleigh_awgn', 'rayleigh_cfo_awgn']:
        print(f'\n--- {attack.upper()} / {channel} ---')
        !python evaluate.py \
          --model_path experiments/results/baseline_cnn/best_model.pth \
          --data_path {DATA_2016} \
          --eval_mode attack_snr \
          --attack_type {attack} \
          --channel_type {channel} \
          --rho 0.01 \
          --pgd_steps 10 \
          --num_trials 10 \
          --output_dir experiments/results/baseline_cnn \
          --device auto \
          --seed 42

In [ ]:
# 8. Phase 2: Attack vs Perturbation Budget
for attack in ['fgsm', 'pgd']:
    print(f'\n--- Budget sweep: {attack.upper()} ---')
    !python evaluate.py \
      --model_path experiments/results/baseline_cnn/best_model.pth \
      --data_path {DATA_2016} \
      --eval_mode attack_budget \
      --attack_type {attack} \
      --channel_type awgn \
      --pgd_steps 10 \
      --num_trials 10 \
      --output_dir experiments/results/baseline_cnn \
      --device auto \
      --seed 42

In [ ]:
# 9. Phase 3: Defense Training & Evaluation (uses tuned defense configs when available)
run_tuned_defense_training(
    data_path=DATA_2016,
    dataset_version='2016.10a',
    num_classes=11,
    input_length=128,
    output_dir='experiments/results',
)

In [ ]:
!ls -la experiments/results/*/best_model.pth

In [ ]:
# 10. Phase 4: Per-Class Vulnerability Analysis
!python eval_per_class.py \
  --model_path experiments/results/baseline_cnn/best_model.pth \
  --data_path {DATA_2016} \
  --snr 0 10 18 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results \
  --device auto

In [ ]:
# 11. Generate Figures
!python plot_results.py \
  --results_dir experiments/results \
  --output_dir experiments/figures

In [ ]:
# 12. View results
import pandas as pd

print('=== Defense Comparison ===')
df = pd.read_csv('experiments/results/defense_comparison.csv')
print(df.to_string(index=False))

print('\n=== Per-Class Vulnerability (SNR=10) ===')
df2 = pd.read_csv('experiments/results/per_class_vulnerability.csv')
df2_10 = df2[df2['snr'] == 10].sort_values('asr', ascending=False)
print(df2_10[['modulation', 'clean_acc', 'robust_acc', 'asr']].to_string(index=False))

In [ ]:
# 13. Display figures
import os
from IPython.display import Image, display
import glob

for fig_path in sorted(glob.glob('experiments/figures/*.png')):
    print(f'\n{os.path.basename(fig_path)}')
    display(Image(filename=fig_path, width=600))

---

# RadioML 2018.01A Experiments

**24 modulations, 1024 I/Q samples, 20GB HDF5 dataset**

The 2018 dataset uses lazy HDF5 loading so it won't OOM. Training takes longer due to larger sequences and `num_workers=0` (required for h5py).

**Estimated time on A100: ~2-3 hours total**

In [ ]:
# 14. Check if 2018 dataset exists (paths on Google Drive — match DRIVE_PROJECT in Setup)
import os

DRIVE_PROJECT = "/content/drive/MyDrive/ew_project/adversarial-rf-robustness"
possible_paths = [
    os.path.join(DRIVE_PROJECT, "data/raw/GOLD_XYZ_OSC.0001_1024.hdf5"),
    os.path.join(DRIVE_PROJECT, "data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5"),
]

DATA_2018 = None
for p in possible_paths:
    if os.path.isfile(p):
        DATA_2018 = p
        size_gb = os.path.getsize(p) / 1e9
        print(f"Found 2018 dataset at: {p} ({size_gb:.1f} GB)")
        break

if DATA_2018 is None:
    print("2018 dataset not found on Drive under data/raw/.")
    print("Upload GOLD_XYZ_OSC.0001_1024.hdf5 to data/raw/ (or archive-3/) on Drive.")
    DATA_2018 = possible_paths[0]

print(f"\nUsing: {DATA_2018}")
RESULTS_2018 = "experiments/results_2018"

In [ ]:
# 15. Smoke test for 2018 dataset
!python smoke_test.py \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024

In [ ]:
# 16. Phase 0 (2018): Baseline CNN Training (uses tuned params when available)
!python train.py \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --epochs {BASELINE_2018_CFG['epochs']} \
  --batch_size {BASELINE_2018_CFG['batch_size']} \
  --lr {BASELINE_2018_CFG['lr']} \
  --weight_decay {BASELINE_2018_CFG['weight_decay']} \
  --seed 42 \
  --save_dir {RESULTS_2018}/baseline_cnn \
  --device auto

In [ ]:
# 17. Phase 1 (2018): Clean Accuracy vs SNR
!python evaluate.py \
  --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --eval_mode clean_snr \
  --num_trials 10 \
  --output_dir {RESULTS_2018}/baseline_cnn \
  --device auto \
  --seed 42

In [ ]:
# 18. Phase 2 (2018): Attack evaluations (PGD + FGSM, AWGN only)
for attack in ['pgd', 'fgsm']:
    print(f'\n--- {attack.upper()} / awgn (2018) ---')
    extra = '--pgd_steps 10' if attack == 'pgd' else ''
    !python evaluate.py \
      --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
      --data_path {DATA_2018} \
      --dataset_version 2018.01a \
      --num_classes 24 \
      --input_length 1024 \
      --eval_mode attack_snr \
      --attack_type {attack} \
      --channel_type awgn \
      --rho 0.01 \
      {extra} \
      --num_trials 10 \
      --output_dir {RESULTS_2018}/baseline_cnn \
      --device auto \
      --seed 42

In [ ]:
# 19. Phase 2 (2018): Attack vs Perturbation Budget
!python evaluate.py \
  --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --eval_mode attack_budget \
  --attack_type pgd \
  --channel_type awgn \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir {RESULTS_2018}/baseline_cnn \
  --device auto \
  --seed 42

In [ ]:
#data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5
DATA_2018 = "/content/drive/MyDrive/ew_project/adversarial-rf-robustness/data/raw/archive-3/GOLD_XYZ_OSC.0001_1024.hdf5"

RESULTS_2018 = "experiments/results_2018"

In [ ]:
!find /content/drive/MyDrive -name "*.hdf5" 2>/dev/null

In [ ]:
# 20. Phase 3 (2018): Defense Training & Evaluation (uses tuned defense configs when available)
run_tuned_defense_training(
    data_path=DATA_2018,
    dataset_version='2018.01a',
    num_classes=24,
    input_length=1024,
    output_dir=RESULTS_2018,
)

In [ ]:
# Fix: Remount Google Drive
from google.colab import drive
import os

# Force unmount first
drive.flush_and_unmount()

# Remount
drive.mount('/content/drive', force_remount=True)

# Verify path and change directory back to project
CODE_DIR = "/content/ew_project/adversarial-rf-robustness"
if os.path.isdir(CODE_DIR):
    os.chdir(CODE_DIR)
    print(f"\nRemounted; cwd: {os.getcwd()}")
    print("Files:", os.listdir("."))
else:
    print("Run the Setup cell first to clone the repo.")

In [ ]:
# 21. Phase 4 (2018): Per-Class Vulnerability Analysis
!python eval_per_class.py \
  --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --snr 0 10 18 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir {RESULTS_2018} \
  --device auto

In [ ]:
# 22. Generate Figures (2018)
!python plot_results.py \
  --results_dir {RESULTS_2018} \
  --output_dir experiments/figures_2018

In [ ]:
# 23. View 2018 results
import pandas as pd

print('=== 2018 Defense Comparison ===')
try:
    df = pd.read_csv(f'{RESULTS_2018}/defense_comparison.csv')
    print(df.to_string(index=False))
except FileNotFoundError:
    print('defense_comparison.csv not found (run Phase 3 first)')

print('\n=== 2018 Per-Class Vulnerability (SNR=10) ===')
try:
    df2 = pd.read_csv(f'{RESULTS_2018}/per_class_vulnerability.csv')
    df2_10 = df2[df2['snr'] == 10].sort_values('asr', ascending=False)
    print(df2_10[['modulation', 'clean_acc', 'robust_acc', 'asr']].to_string(index=False))
except FileNotFoundError:
    print('per_class_vulnerability.csv not found (run Phase 4 first)')

In [ ]:
# 24. Display 2018 figures
from IPython.display import Image, display
import glob

for fig_path in sorted(glob.glob('experiments/figures_2018/*.png')):
    print(f'\n{os.path.basename(fig_path)}')
    display(Image(filename=fig_path, width=600))

print('\n' + '=' * 60)
print(' ALL 2018 EXPERIMENTS COMPLETE')
print('=' * 60)
print(f'Results: {RESULTS_2018}/')
print('Figures: experiments/figures_2018/')

In [ ]:
# Cell 1: PGD Attack vs SNR — Rayleigh fading (~2-3 hrs on A100)
!python evaluate.py \
  --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --eval_mode attack_snr \
  --attack_type pgd \
  --channel_type rayleigh_awgn \
  --rho 0.01 \
  --pgd_steps 10 \
  --output_dir {RESULTS_2018}/baseline_cnn \
  --device auto \
  --seed 42 \
  --num_trials 10

In [ ]:
# Cell 2: PGD Attack vs SNR — Rayleigh+CFO (~2-3 hrs on A100)
!python evaluate.py \
  --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --eval_mode attack_snr \
  --attack_type pgd \
  --channel_type rayleigh_cfo_awgn \
  --rho 0.01 \
  --pgd_steps 10 \
  --output_dir {RESULTS_2018}/baseline_cnn \
  --device auto \
  --seed 42 \
  --num_trials 10

---

# Transferability Analysis

Evaluate whether adversarial examples crafted against the baseline model transfer to defended models.

In [ ]:
# Transferability Analysis (2016.10a)
!python eval_transferability.py \
  --source_path experiments/results/baseline_cnn/best_model.pth \
  --source_name baseline \
  --target_path \
    experiments/results/channel_aug/best_model.pth \
    experiments/results/adv_train/best_model.pth \
    experiments/results/noise_inject/best_model.pth \
  --target_names "Channel Aug." "Adv. Training" "Noise Inj." \
  --data_path {DATA_2016} \
  --dataset_version 2016.10a \
  --snr 0 10 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir experiments/results \
  --device auto

In [ ]:
# Transferability Analysis (2018.01a)
RESULTS_2018 = "experiments/results_2018"

!python eval_transferability.py \
  --source_path {RESULTS_2018}/baseline_cnn/best_model.pth \
  --source_name baseline \
  --target_path \
    {RESULTS_2018}/channel_aug/best_model.pth \
    {RESULTS_2018}/adv_train/best_model.pth \
    {RESULTS_2018}/noise_inject/best_model.pth \
    {RESULTS_2018}/adv_train_channel/best_model.pth \
  --target_names "Channel Aug." "Adv. Training" "Noise Inj." "Adv.+Chan. Aug." \
  --data_path {DATA_2018} \
  --dataset_version 2018.01a \
  --num_classes 24 \
  --input_length 1024 \
  --snr 0 10 \
  --rho 0.01 \
  --attack pgd \
  --pgd_steps 10 \
  --num_trials 10 \
  --output_dir {RESULTS_2018} \
  --device auto

In [ ]:
# View Transferability Results
import pandas as pd

print('=== 2016 Transferability ===')
try:
    df = pd.read_csv('experiments/results/transferability_analysis.csv')
    print(df.to_string(index=False))
except FileNotFoundError:
    print('Not found — run 2016 transferability cell first.')

print('\n=== 2018 Transferability ===')
try:
    df = pd.read_csv('experiments/results_2018/transferability_analysis.csv')
    print(df.to_string(index=False))
except FileNotFoundError:
    print('Not found — run 2018 transferability cell first.')

---

# Channel-Aware Attack Comparison (freeze_channel)

Compare PGD attacks with and without channel state information (CSI):
- **No CSI** (default): Each PGD step sees an independent channel realization
- **Perfect CSI** (`--freeze-channel`): All PGD steps use the same fixed channel realization

In [ ]:
# Freeze-Channel PGD Attacks (2016)
for channel in ['rayleigh_awgn', 'rayleigh_cfo_awgn']:
    print(f'\n--- PGD freeze-channel / {channel} ---')
    !python evaluate.py \
      --model_path experiments/results/baseline_cnn/best_model.pth \
      --data_path {DATA_2016} \
      --eval_mode attack_snr \
      --attack_type pgd \
      --channel_type {channel} \
      --rho 0.01 \
      --pgd_steps 10 \
      --freeze-channel \
      --num_trials 10 \
      --output_dir experiments/results/baseline_cnn \
      --device auto \
      --seed 42

In [ ]:
# Freeze-Channel PGD Attacks (2018)
RESULTS_2018 = "experiments/results_2018"

for channel in ['rayleigh_awgn', 'rayleigh_cfo_awgn']:
    print(f'\n--- PGD freeze-channel / {channel} (2018) ---')
    !python evaluate.py \
      --model_path {RESULTS_2018}/baseline_cnn/best_model.pth \
      --data_path {DATA_2018} \
      --dataset_version 2018.01a \
      --num_classes 24 \
      --input_length 1024 \
      --eval_mode attack_snr \
      --attack_type pgd \
      --channel_type {channel} \
      --rho 0.01 \
      --pgd_steps 10 \
      --freeze-channel \
      --num_trials 10 \
      --output_dir {RESULTS_2018}/baseline_cnn \
      --device auto \
      --seed 42

---

# Per-Class Precision, Recall & F1

Per-class precision, recall, and F1-score are computed automatically by `eval_per_class.py` and saved to `per_class_prf_clean.csv` and `per_class_prf_adv.csv`.

In [ ]:
# View Per-Class PRF Results
import pandas as pd

for label, results_dir in [('2016', 'experiments/results'), ('2018', 'experiments/results_2018')]:
    print(f'\n=== {label} Per-Class PRF (Clean) ===')
    try:
        df = pd.read_csv(f'{results_dir}/per_class_prf_clean.csv')
        print(df.to_string(index=False))
    except FileNotFoundError:
        print(f'Not found — run per-class eval for {label} first.')

    print(f'\n=== {label} Per-Class PRF (Adversarial) ===')
    try:
        df = pd.read_csv(f'{results_dir}/per_class_prf_adv.csv')
        print(df.to_string(index=False))
    except FileNotFoundError:
        print(f'Not found — run per-class eval for {label} first.')